# M3 end-to-end pipeline (P1): from a photo or a clip to a verdict and first aid

We run the whole app chain, from a photo or a clip to what the screen shows, through one call,
`pipeline.run_pipeline`. This is P1: do the stages built so far join up, how long does each node take, and does the
chain give the same answers as the training run? The last section renders one clip the way the app shows it.

```mermaid
flowchart LR
  A[Photo] --> B[Preprocess: RGB, exposure fix, letterbox, ImageNet norm]
  V[Video clip] --> S[video.schedule: the newest frame, at most 10 a second] -->|each kept frame| B
  B --> C{Front end: the run's F-mode}
  C -->|F0| D[Whole photo]
  C -->|F1| E[D1 detector box crop]
  C -->|box_conf below tau_snake| X[No snake found: SnakeID skipped]
  D --> G[SnakeID: encoder and embedding]
  E --> G
  G --> H[Species head: top 3]
  G --> P[Class prototypes: best cosine against tau_unknown]
  G --> I[Genus, family and WHO heads]
  H --> J{Decision: decision.verdict}
  P --> J
  I --> J
  X --> J
  J -->|photo| R{Verdict}
  J -->|clip, each kept frame| W[4-of-6 vote; no majority shows we don't know] --> R
  R -->|WHO 1 or 2 likely, viper or elapid, guarded genus| K[Dangerous: keep away]
  R -->|confident, not listed| L[Not on the WHO list, still keep away]
  R -->|no snake, far from every prototype, unsure, frames disagree| M[We don't know: keep away]
  K --> N[Response: species_card and the R0 lookup of WHO passages]
  L --> N
  M --> N
  N --> O[Screen: verdict, top 3, reason, quoted guidance with its page]
```

Each node logs a short summary of its output and its milliseconds. The top 3 come from the temperature-scaled species
head; the prototypes give the best cosine that tau_unknown reads. A clip runs the first four nodes on the frames
`video.schedule` keeps at 10 frames a second. The screen shows the 4-of-6 vote on the per-frame verdicts
(`video.vote_k_of_n`): "we don't know" until 4 of the last 6 frames agree and whenever they stop agreeing, then the
species card and passages of that verdict.

**Inputs**, local paths beside the repo: a run folder (`RUN_DIR`: `best.pt`, `config.yaml`, `prototypes.npz`,
`val_outputs.npz`) and its thresholds (`DECISION_FILE`), the front-end weights (`FRONTEND_DIR`: `winners.json` and the
D1 detector), the photo sample (`SAMPLE_DIR`) and the WHO guideline PDF at `refs/who_searo_2016.pdf` (or
`SERPENTGUARD_WHO_PDF`). `load_models` refuses a decision file that is not the run's own: `decision.decide` on the
run's `val_outputs.npz` at the file's thresholds must give the file's validation verdict counts.

**Outputs:** `results/m3/p1_final/` (committed) holds `photos.jsonl` and `clips.jsonl` (one line per photo or clip,
with every node's log), `examples/` (8 example outputs: 2 per verdict and 2 failures) and `summary.json`. They hold
passage ids and pages, never passage text, and no image: only each photo's path, licence and attribution from the
manifest. The same 8 examples with their passage text go to `TEXT_OUT`, outside the repo, and are never committed.
The demo writes its insights with ids and pages only to `results/m3/demo/`; the annotated video and the insights with
passage text go to `DEMO_OUT`, outside the repo. No cell prints guideline text.

**Run:** locally on the CPU, from the `notebooks/` folder. `RUN_DIR`, `DECISION_FILE` and `FRONTEND_DIR` choose the
model and front end, `N_PHOTOS`, `N_CLIPS` and `SEED` the sample, and the `DEMO_` parameters the demo clip.

**The saved outputs** come from 100 validation photos of the local sample (`use == "val"`, classes in turn, seed 42)
and 5 clips made from the first 5 of them by `video.make_clip` (M1 section 3.4's recipe, seeds 42 to 46), with the
final model H1_top1_s1, its shipped decision file and the D1 detector (front end F1): a photo whose best box is under
`tau_snake` gets "no snake found".

In [1]:
from pathlib import Path

WORKSPACE = Path("..").resolve().parent   # the folder holding the repo and kaggle/
RUN_DIR = WORKSPACE / "kaggle" / "m3_runs" / "stageC" / "final_candidates" / "top1_s1" / "H1_top1_s1"  # best.pt, config
DECISION_FILE = Path("../results/m3/final/H1_top1_s1/decision_metrics.json")  # the run's shipped thresholds
FRONTEND_DIR = WORKSPACE / "kaggle" / "m3_runs" / "frontends" / "weights" / "frontends"  # winners.json and D1/
SAMPLE_DIR = WORKSPACE / "kaggle" / "m3_runs" / "local_sample" / "out2" / "local_sample"
N_PHOTOS = 100
N_CLIPS = 5
SEED = 42
OUT = Path("../results/m3/p1_final")             # committed: ids, pages, verdicts and logs; no passage text
TEXT_OUT = WORKSPACE / "outputs" / "p1_final"    # never committed: the examples with their passage text
DEMO_PHOTO = SAMPLE_DIR / "data" / "val" / "bungarus_caeruleus" / "inaturalist_754593781.jpg"
DEMO_VIDEO = None                                # a snake video to annotate instead of filming DEMO_PHOTO
DEMO_SEED = 42
DEMO_MAX_SECONDS = None
DEMO_OUT = WORKSPACE / "outputs" / "demo"        # never committed: the annotated video and the insights with text
DEMO_EXAMPLE = Path("../results/m3/demo/final_bungarus_caeruleus_insights.json")  # committed: ids and pages only

In [2]:
import json
import sys
import time
from collections import Counter

SRC = Path("../src")
if SRC.is_dir() and str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

import numpy as np
import pandas as pd

from serpentguard.data import IMAGE_FOLDER, class_index
from serpentguard.decision import decide, repo_tables
from serpentguard.metrics import VERDICTS
from serpentguard.pipeline import CLIP_SCOPE, FAILURE, NODES, load_models, p1_record, pick_examples, run_pipeline
from serpentguard.respond import LookupRetriever, build_kb, find_who_pdf
from serpentguard.run_outputs import library_versions, subset_hash, write_json
from serpentguard.train import VAL_OUTPUTS
from serpentguard.video import load_raw_rgb, make_clip, round_robin

PDF = find_who_pdf()
assert PDF is not None, "put the guideline at refs/who_searo_2016.pdf beside the repo or set SERPENTGUARD_WHO_PDF"
models, thresholds = load_models(RUN_DIR, DECISION_FILE, FRONTEND_DIR)
tables = models.tables or repo_tables()
lookup = LookupRetriever(build_kb(PDF))
decision_report = json.loads(DECISION_FILE.read_text())
print(f"{DECISION_FILE.name} checked against {RUN_DIR.name}'s own validation outputs: decide() reproduces its counts "
      f"{decision_report['verdicts']['val']['total']} = {[decision_report['verdicts']['val'][v] for v in VERDICTS]} "
      f"{list(VERDICTS)}")
print(f"run {RUN_DIR.name}: front end {models.front_end}, input {models.input_size}, backend {models.backend}, "
      f"detector {'yes' if models.detector is not None else 'none'}, tau_snake {models.tau_snake}")
print(thresholds)

decision_metrics.json checked against H1_top1_s1's own validation outputs: decide() reproduces its counts 17151 = [7778, 7737, 1636] ['dangerous', 'not_listed', 'unknown']
run H1_top1_s1: front end F1, input 320, backend torch, detector yes, tau_snake 0.102
Thresholds(temperature=1.9086784115427977, temperature_who=0.9690096288687304, theta_d=0.05660591583161018, theta_who=inf, tau_unknown=0.40982445157291975, theta_conf=0.360998147721644, theta_v=0.5, theta_genus=0.5)


## 1. The photos

The validation rows of the local sample's manifest, taken by `video.round_robin`: the classes in a seeded random
order, one photo each in turn, so 100 photos cover 100 different classes.

In [3]:
manifest = pd.read_csv(SAMPLE_DIR / "manifest.csv", dtype={"observer_id": str})
val = manifest[manifest["use"].eq("val") & manifest["split"].eq("val")]
classes = pd.read_csv("../data/classes.csv")
val = val.assign(class_id=val["species"].map(class_index(classes)))
assert val["class_id"].notna().all(), "every validation species must be in classes.csv"
photos = round_robin(val, N_PHOTOS, SEED)
photos = photos.assign(class_id=photos["class_id"].astype(int),
                       image_file=[str(SAMPLE_DIR / IMAGE_FOLDER / path) for path in photos["path"]])
dangerous = tables.dangerous[photos["class_id"].to_numpy()]
listed = tables.who[photos["class_id"].to_numpy()] > 0
print(f"{len(photos)} photos from {photos['species'].nunique()} classes: {int(dangerous.sum())} of a dangerous "
      f"species ({int(listed.sum())} WHO-listed), {int((~dangerous).sum())} not; licences "
      f"{photos['license'].value_counts().to_dict()}")

100 photos from 100 classes: 26 of a dangerous species (12 WHO-listed), 74 not; licences {'cc-by-nc': 80, 'cc-by': 14, 'cc-by-nc-sa': 5, 'cc-by-sa': 1}


## 2. P1 on 100 photos

One untimed call first, so the timed calls don't carry the first call's one-off costs. Each record holds the
photo's path, licence and attribution, the expected verdict (dangerous for a danger-table species, not listed
otherwise; "we don't know" is never expected) and any failure: `missed_danger` (a danger-table snake: WHO-listed,
viper or elapid, or keelback, not called dangerous, "we don't know" included) or `false_alarm` (a snake outside the
table called dangerous). "We don't know" on a species outside the table is the cautious answer, not a failure.

In [4]:
run_pipeline(photos["image_file"].iloc[0], models, thresholds, lookup)
results, records = {}, []
started = time.time()
for photo in photos.to_dict("records"):
    results[photo["path"]] = run_pipeline(photo["image_file"], models, thresholds, lookup)
    records.append(p1_record(photo, results[photo["path"]], tables))
print(f"{len(records)} photos in {time.time() - started:.1f} s")


def node_ms(records, unit):
    """Median ms per node over all photos or clip frames (clip-level nodes apart), and the median total per unit."""
    rows = [{"node": f"{entry['node']} ({CLIP_SCOPE})" if entry.get("frame") == CLIP_SCOPE else entry["node"],
             "ms": entry["ms"]} for record in records for entry in record["log"]]
    medians = pd.DataFrame(rows).groupby("node", sort=False)["ms"].median()
    return {**medians.round(3).to_dict(), f"total per {unit}": round(float(np.median([r["total_ms"] for r in records])), 3)}


photo_ms = node_ms(records, "photo")
table = pd.DataFrame(records)
print("median ms per node, photos:", photo_ms)
print(pd.crosstab(table["expected"], table["verdict"], margins=True).to_string())
print("failures:", table["failure"].value_counts().to_dict())
print("reasons for we don't know:", Counter(r for rs in table.loc[table["verdict"] == "unknown", "reason"] for r in rs))

100 photos in 9.9 s
median ms per node, photos: {'preprocess': 7.206, 'front_end': 15.0, 'snakeid': 72.502, 'decision': 0.376, 'response': 0.014, 'total per photo': 95.312}
verdict     dangerous  not_listed  unknown  All
expected                                       
dangerous          25           0        1   26
not_listed         21          40       13   74
All                46          40       14  100
failures: {'false_alarm': 21, 'missed_danger': 1}
reasons for we don't know: Counter({'unclear_between_species': 14, 'unfamiliar': 1})


### Check against the run's own saved outputs

The training run saved every validation photo's outputs on Kaggle's GPU (`val_outputs.npz`). `decision.decide` on
those rows, at the same thresholds and prototypes, should give the verdicts and top-1 species P1 gave on this CPU from
the photo files, given the same no-snake flags; a small share can differ, since Kaggle's run read the photos at its
own resolution in fp16, and the largest difference in the top-1 probability shows how close they are. The
negative control pairs each P1 photo with the saved row of the next photo, so its agreement must fall to about what
two unrelated photos share.

In [5]:
with np.load(RUN_DIR / VAL_OUTPUTS) as saved:
    row_of = {str(path): row for row, path in enumerate(saved["paths"])}
    matched = [index for index, record in enumerate(records) if record["path"] in row_of]
    rows = [row_of[records[index]["path"]] for index in matched]
    outputs = {key: saved[key][rows] for key in ("embedding", "species_logits", "who_logits", "genus_logits",
                                                    "family_logits")}
snake_flags = np.array([records[index]["snake_found"] for index in matched])
saved_decisions = decide(outputs, thresholds, models.prototypes, tables, snake_found=snake_flags)
p1_verdicts = np.array([records[index]["verdict"] for index in matched])
with_snake = [index for index in matched if records[index]["snake_found"]]
p1_top1 = np.array([records[index]["top3"][0][0] for index in with_snake])
p1_top1_prob = np.array([results[records[index]["path"]].top3[0][1] for index in with_snake])
saved_top1 = tables.species[saved_decisions.top1[snake_flags]]
saved_top1_prob = saved_decisions.top1_prob[snake_flags]
cross_check = {"photos_matched": len(matched),
               "verdict_agreement": float(np.mean(saved_decisions.label == p1_verdicts)),
               "top1_agreement": float(np.mean(saved_top1 == p1_top1)),
               "control_verdict_agreement": float(np.mean(np.roll(saved_decisions.label, 1) == p1_verdicts)),
               "control_top1_agreement": float(np.mean(np.roll(saved_top1, 1) == p1_top1)),
               "max_top1_prob_difference": float(np.abs(saved_top1_prob - p1_top1_prob).max()),
               "snake_flags_passed": int(snake_flags.sum())}
print(json.dumps(cross_check, indent=1))

{
 "photos_matched": 100,
 "verdict_agreement": 0.98,
 "top1_agreement": 0.98,
 "control_verdict_agreement": 0.37,
 "control_top1_agreement": 0.01,
 "max_top1_prob_difference": 0.18726019626040102,
 "snake_flags_passed": 100
}


## 3. P1 on 5 clips

Each clip is 10 seconds at 15 FPS from one photo as stored (`video.load_raw_rgb`, then `video.make_clip`, as V1
makes them; the leaves go over a central ellipse, since the masks are not on this machine). The pipeline reads the
frames `video.schedule` keeps at 10 a second, and the screen shows the 4-of-6 vote. The clip frames are made in memory
and dropped after each clip; none is saved.

Clip stability counts what the screen shows, and that includes the "we don't know" placeholder shown before the first
majority: a placeholder that lasts a second counts as a stable label and as a flip when the real verdict replaces
it. `first_majority_at` is when the first real verdict appears.

In [6]:
clip_records = []
started = time.time()
for index, photo in enumerate(photos.head(N_CLIPS).to_dict("records")):
    frames = make_clip(load_raw_rgb(photo["image_file"]), seed=SEED + index)
    result = run_pipeline(frames, models, thresholds, lookup)
    clip_records.append({**p1_record(photo, result, tables), "clip_seed": SEED + index})
    del frames
print(f"{len(clip_records)} clips in {time.time() - started:.1f} s")
clip_ms = node_ms(clip_records, "clip")
print("median ms per node, clip frames:", clip_ms)
clip_table = pd.DataFrame([{"species": r["species"], "expected": r["expected"], "final": r["verdict"],
                            "failure": r["failure"], **{key: r["clip"][key] for key in (
                                "processed", "effective_fps", "flips", "time_to_stable", "first_majority_at", "majority")},
                            "per_frame": r["clip"]["per_frame"]} for r in clip_records])
print(clip_table.to_string(index=False, float_format="%.2f"))

5 clips in 45.5 s
median ms per node, clip frames: {'preprocess': 4.035, 'front_end': 14.485, 'snakeid': 62.652, 'decision': 0.359, 'decision (clip)': 0.232, 'response (clip)': 0.013, 'total per clip': 8315.412}
                species   expected     final     failure  processed  effective_fps  flips  time_to_stable  first_majority_at  majority                                          per_frame
        Boiga siamensis not_listed dangerous false_alarm        101          10.00      1            0.40               0.40      True                                 {'dangerous': 101}
      Oligodon cinereus not_listed dangerous false_alarm        101          10.00     13            0.10               0.90      True  {'dangerous': 55, 'not_listed': 5, 'unknown': 41}
Sibynophis subpunctatus not_listed   unknown         NaN        101          10.00     14            0.80               0.60     False {'dangerous': 40, 'unknown': 30, 'not_listed': 31}
     Calliophis bibroni  dangerous dangerous

## 4. Examples and saved logs

Two of each verdict (the first in run order that are not failures) and two failures: a missed danger first, the worst
one (a viper or elapid called "not listed" before any other miss), then a false alarm.

In [7]:
for folder in (OUT / "examples", TEXT_OUT / "examples"):
    folder.mkdir(parents=True, exist_ok=True)
    for old in folder.glob("*.json"):
        old.unlink()
for name, rows in (("photos.jsonl", records), ("clips.jsonl", clip_records)):
    (OUT / name).write_text("".join(json.dumps(row) + "\n" for row in rows))
examples = pick_examples(records)
by_path = photos.set_index("path")
for number, example in enumerate(examples, 1):
    name = f"{number:02d}_{example['example']}.json"
    write_json(OUT / "examples" / name, example)
    photo = {**by_path.loc[example["path"]].to_dict(), "path": example["path"]}
    write_json(TEXT_OUT / "examples" / name,
               {**p1_record(photo, results[example["path"]], tables, with_text=True), "example": example["example"]})
print(pd.DataFrame([{"example": e["example"], "species": e["species"], "expected": e["expected"],
                     "verdict": e["verdict"], "failure": e["failure"], "reason": e["reason"],
                     "top1": e["top3"][0] if e["top3"] else None, "passages": [p["id"] for p in e["passages"]]}
                    for e in examples]).to_string(index=False))

   example                 species   expected    verdict       failure                                                         reason                                  top1                                                                                            passages
 dangerous      Calliophis bibroni  dangerous  dangerous           NaN [who_listed_mass, top_species_dangerous, venomous_family_mass]          [Bungarus fasciatus, 0.4788]                  [who-110-01, who-119-04, who-118-04, who-119-03, who-120-02, who-120-05, card-023]
 dangerous      Ophiophagus hannah  dangerous  dangerous           NaN                                              [who_listed_mass]                [Ptyas mucosa, 0.5921]                  [who-110-01, who-119-04, who-118-04, who-119-03, who-120-02, who-120-05, card-008]
not_listed       Oligodon cinereus not_listed not_listed           NaN                                                             [] [Psammodynastes pulverulentus, 0.362] [note-not-li

In [8]:
REPO = Path("..").resolve()


def shown(path):
    """A path as the results record it: relative to the repo inside it, else relative to the workspace."""
    path = Path(path).resolve()
    return str(path.relative_to(REPO if path.is_relative_to(REPO) else WORKSPACE))


summary = write_json(OUT / "summary.json", {
    "run": {"run_dir": RUN_DIR.name, "decision_file": shown(DECISION_FILE), "front_end": models.front_end,
            "frontend_dir": None if FRONTEND_DIR is None else shown(FRONTEND_DIR), "backend": models.backend,
            "onnx_path": None, "input_size": models.input_size,
            "thresholds": vars(thresholds), "tau_snake": models.tau_snake, "device": "cpu"},
    "sample": {"photos": len(photos), "classes": int(photos["species"].nunique()), "seed": SEED,
               "photos_hash": subset_hash(photos["path"]), "clips": len(clip_records),
               "clip_seeds": [r["clip_seed"] for r in clip_records]},
    "definitions": {"expected": "dangerous for a danger-table species, not listed otherwise",
                    "missed_danger": "a danger-table snake (WHO-listed, viper or elapid, keelback) not called dangerous",
                    "false_alarm": "a snake outside the danger table called dangerous",
                    "clip_stability": "counts the we-don't-know placeholder shown before the first majority"},
    "decision_file_check": decision_report["verdicts"]["val"],
    "median_ms": {"photos": photo_ms, "clip_frames": clip_ms},
    "cross_check_with_saved_outputs": cross_check,
    "verdicts": {"photos": dict(Counter(r["verdict"] for r in records)),
                 "by_expected": {expected: dict(Counter(r["verdict"] for r in records if r["expected"] == expected))
                                 for expected in ("dangerous", "not_listed")},
                 "failures": dict(Counter(r["failure"] for r in records if r["failure"])),
                 "clips": dict(Counter(r["verdict"] for r in clip_records))},
    "clips": [{"path": r["path"], "species": r["species"], "expected": r["expected"], "failure": r["failure"],
               "clip_seed": r["clip_seed"], **r["clip"]} for r in clip_records],
    "examples": [{"file": f"{n:02d}_{e['example']}.json", "path": e["path"], "example": e["example"]}
                 for n, e in enumerate(examples, 1)],
    "versions": library_versions(),
})
assert sum(summary["verdicts"]["photos"].values()) == len(photos)
print(json.dumps(summary["verdicts"], indent=1))
print("wrote", sorted(path.name for path in OUT.rglob("*.json*")))

{
 "photos": {
  "dangerous": 46,
  "not_listed": 40,
  "unknown": 14
 },
 "by_expected": {
  "dangerous": {
   "dangerous": 25,
   "unknown": 1
  },
  "not_listed": {
   "dangerous": 21,
   "not_listed": 40,
   "unknown": 13
  }
 },
 "failures": {
  "false_alarm": 21,
  "missed_danger": 1
 },
 "clips": {
  "dangerous": 4,
  "unknown": 1
 }
}
wrote ['01_dangerous.json', '02_dangerous.json', '03_not_listed.json', '04_not_listed.json', '05_unknown.json', '06_unknown.json', '07_failure.json', '08_failure.json', 'box_check.json', 'clips.jsonl', 'photos.jsonl', 'summary.json']


## Reading the result

Every number below is read from this run's results.

In [9]:
table = pd.DataFrame(records)
listed = table[table["who_listed"]]
missed = table[table["failure"] == "missed_danger"]
outside = table[table["expected"] == "not_listed"]
alarms = int((table["failure"] == "false_alarm").sum())
report_alarm = decision_report["verdict_metrics"]["false_alarm_outside_table"]
no_card = [r for r in records if r["verdict"] == "dangerous" and r["card"] is None]
clip_finals = Counter(r["verdict"] for r in clip_records)
clip_alarms = sum(r["failure"] == "false_alarm" for r in clip_records)
first = clip_records[0]["clip"]
reading = [
    f"- It agrees with the training run: on {cross_check['photos_matched']} photos, decide() on the run's saved outputs "
    f"gives the same verdict for {cross_check['verdict_agreement']:.0%} and the same top-1 species for "
    f"{cross_check['top1_agreement']:.0%}, with top-1 probabilities at most {cross_check['max_top1_prob_difference']:.1e} "
    f"apart; the shifted control agrees on {cross_check['control_verdict_agreement']:.0%} and "
    f"{cross_check['control_top1_agreement']:.0%}.",
    f"- Speed in this run: the median photo takes {photo_ms['total per photo']:.1f} ms, SnakeID "
    f"{photo_ms['snakeid']:.1f} ms of it ({models.backend}), preprocessing {photo_ms['preprocess']:.1f} ms, the "
    f"decision {photo_ms['decision']:.2f} ms and the lookup {photo_ms['response']:.2f} ms; SnakeID's median on clip "
    f"frames is {clip_ms['snakeid']:.1f} ms. Other jobs can share this laptop's CPU, so these times show where the "
    f"time goes, not the O6 latency, which V1 measures.",
    f"- WHO-listed photos called dangerous: {int((listed['verdict'] == 'dangerous').sum())} of {len(listed)}. Missed "
    f"danger, {len(missed)} photos: " + "; ".join(f"{r.species} called {r.verdict} (top-1 {r.top3[0][0]})"
                                                   for r in missed.itertuples()) + ".",
    f"- False alarms: {alarms} of the {len(outside)} photos outside the danger table ({alarms / len(outside):.2f}); "
    f"the decision report gives {report_alarm:.2f} on the whole validation split at theta_d = {thresholds.theta_d:.3f}.",
    f"- Clips: {dict(clip_finals)} at the end, {clip_alarms} of them false alarms. Each read {first['processed']} of "
    f"{first['frames']} frames ({first['skipping']}). The first real verdict appeared at "
    + ", ".join(f"{r['clip']['first_majority_at']:.1f} s" for r in clip_records)
    + "; flips and time to stable count the we-don't-know placeholder before it.",
    f"- The species card: {len(no_card)} dangerous photos show no card, because no top-3 species is in the danger "
    f"table" + (f" (first: {no_card[0]['species']}, top-1 {no_card[0]['top3'][0][0]})" if no_card else "")
    + "; a dangerous screen never shows a card that says it is not on the WHO list.",
    "- Front end: " + ("no detector, so no photo gets 'no snake found'; with FRONTEND_DIR set, a photo whose best box "
                       "is below tau_snake skips SnakeID and gets it." if models.detector is None else
                       f"{sum(not r['snake_found'] for r in records)} photos had no snake found."),
]
print("\n".join(reading))

- It agrees with the training run: on 100 photos, decide() on the run's saved outputs gives the same verdict for 98% and the same top-1 species for 98%, with top-1 probabilities at most 1.9e-01 apart; the shifted control agrees on 37% and 1%.
- Speed in this run: the median photo takes 95.3 ms, SnakeID 72.5 ms of it (torch), preprocessing 7.2 ms, the decision 0.38 ms and the lookup 0.01 ms; SnakeID's median on clip frames is 62.7 ms. Other jobs can share this laptop's CPU, so these times show where the time goes, not the O6 latency, which V1 measures.
- WHO-listed photos called dangerous: 12 of 12. Missed danger, 1 photos: Ovophis monticola called unknown (top-1 Boiga multomaculata).
- False alarms: 21 of the 74 photos outside the danger table (0.28); the decision report gives 0.30 on the whole validation split at theta_d = 0.057.
- Clips: {'dangerous': 4, 'unknown': 1} at the end, 3 of them false alarms. Each read 101 of 150 frames (the 10 frames-per-second cap). The first real verdic

## 5. Demo: one clip as the app shows it

The same pipeline on one clip, drawn frame by frame: the screen's verdict in a banner of its colour (with the likeliest
genus when the screen says "we don't know"), the top 3, the detector's box and its confidence, and a corner line with
the frame's own verdict and milliseconds; a skipped frame shows the last processed one. The insights give the final
verdict and when it first became stable, the flips before and after the 4-of-6 vote, the species most often on top,
the species card and passages with their pages, and the time per node.

We have no real snake video yet, so the demo films `DEMO_PHOTO`, a validation photo, with V1's clip recipe; set
`DEMO_VIDEO` to annotate a real video instead. The video is H.264 where OpenCV can write it, MPEG-4 Part 2 otherwise.
It stays outside the repo, since it is made from someone else's photo under its own licence.

In [10]:
import json
import math
from bisect import bisect_right
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from serpentguard.config import load_config
from serpentguard.data import IMAGE_FOLDER, class_index
from serpentguard.decision import GENUS_LISTED_NOTE, POINT, REASON_TEXT, expected_verdict, repo_tables
from serpentguard.frontend import D1, WINNERS_FILE
from serpentguard.medical_importance import CLASSES_CSV
from serpentguard.metrics import DANGEROUS, NOT_LISTED, UNKNOWN
from serpentguard.pipeline import CLIP_SCOPE, FRONT_END, MS_DECIMALS, PER_FRAME_NODES, result_record, run_pipeline
from serpentguard.run_files import CHECKPOINT_FILE, CONFIG_FILE
from serpentguard.run_outputs import file_sha256
from serpentguard.video import (CLIP_FPS, CLIP_MAX_SIDE, CLIP_SECONDS, SEED, STABLE_SECONDS, load_raw_rgb, phone_frame,
                                synthesize_clip, video_metrics)

ANNOTATED_FILE, INSIGHTS_JSON, INSIGHTS_MD = "annotated.mp4", "insights.json", "insights.md"
CODECS = ("avc1", "mp4v")  # H.264 plays in browsers and slides; MPEG-4 Part 2 only where OpenCV cannot open it
VIDEO, PHOTO = "video", "photo"
EVEN = 2                   # H.264 frames have even sides: the writer drops an odd last row or column
PERCENT = "{:.0%}"

VERDICT_COLOUR = {DANGEROUS: (200, 30, 30), NOT_LISTED: (215, 120, 0), UNKNOWN: (85, 85, 105)}  # RGB
TEXT_COLOUR = (255, 255, 255)
BOX_COLOUR, NO_SNAKE_BOX_COLOUR = (255, 230, 0), (150, 150, 150)
PANEL_COLOUR, PANEL_ALPHA = np.array((0, 0, 0), np.float32), 0.6
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE_AT_REFERENCE, REFERENCE_WIDTH, MIN_FONT_SCALE = 0.6, 640, 0.4
CORNER_SCALE = 0.9         # the corner line's text, relative to the rest
LINE_SPACING = 1.6         # line height over the text's own height
SIZING_TEXT = "Ag"         # a capital and a descender: the height every line is spaced by
MARGIN_SHARE = 0.012       # of the frame's long side
BOX_WIDTH_SHARE, MIN_BOX_WIDTH = 0.005, 2


@dataclass(frozen=True)
class Footage:
    """The frames (uint8 RGB), their frame rate and where they came from."""
    frames: list
    fps: float
    source: dict


@dataclass(frozen=True)
class Overlay:
    """What one annotated frame shows."""
    colour: tuple
    banner: tuple
    top3_header: str
    top3: tuple
    corner: str
    box: tuple | None = None
    box_conf: float | None = None
    snake_found: bool = True


def read_video(path, max_seconds=None, max_side=CLIP_MAX_SIDE):
    """The video's frames (RGB, long side at most max_side) and its own frame rate, up to max_seconds."""
    capture = cv2.VideoCapture(str(path))
    fps = capture.get(cv2.CAP_PROP_FPS) if capture.isOpened() else 0.0
    if not (math.isfinite(fps) and fps > 0):
        raise ValueError(f"OpenCV cannot read {path}: no frame rate")
    frames, size = [], None
    limit = math.inf if max_seconds is None else round(max_seconds * fps)
    while len(frames) < limit:
        ok, bgr = capture.read()
        if not ok:
            break
        size = size or [int(bgr.shape[1]), int(bgr.shape[0])]
        frames.append(cv2.cvtColor(phone_frame(bgr, max_side), cv2.COLOR_BGR2RGB))
    capture.release()
    if not frames:
        raise ValueError(f"OpenCV cannot read {path}: no frames")
    return Footage(frames, float(fps), {"kind": VIDEO, "file": Path(path).name, "source_size": size,
                                        "max_seconds": max_seconds})


def photo_source(photo):
    """The photo's row in the nearest manifest.csv above it; an unlisted photo is refused, its licence unknown."""
    photo = Path(photo).resolve()
    manifest = next((folder / "manifest.csv" for folder in photo.parents if (folder / "manifest.csv").is_file()), None)
    if manifest is None:
        raise FileNotFoundError(f"no manifest.csv above {photo}: its licence and attribution are unknown")
    table = pd.read_csv(manifest, dtype=str, keep_default_na=False)
    listed = [row for row in table.to_dict("records") if (manifest.parent / IMAGE_FOLDER / row["path"]).resolve() == photo]
    if not listed:
        raise ValueError(f"{manifest} does not list {photo}: its licence and attribution are unknown")
    return listed[0]


def photo_footage(photo, seed=SEED, max_seconds=None):
    """A clip filmed from one sample photo with V1's recipe, at most CLIP_SECONDS long, and the photo's source."""
    row = photo_source(photo)
    raw = load_raw_rgb(photo)
    seconds = CLIP_SECONDS if max_seconds is None else min(CLIP_SECONDS, max_seconds)
    clip = synthesize_clip(raw, seconds=seconds, seed=seed)
    class_id = class_index(pd.read_csv(CLASSES_CSV)).get(row["species"])
    expected = None if class_id is None else expected_verdict(class_id, repo_tables())
    keys = ("path", "species", "license", "attribution", "url")
    return Footage(clip.frames, float(CLIP_FPS), {
        "kind": PHOTO, "photo": {**{key: row.get(key, "") for key in keys}, "expected_verdict": expected},
        "source_size": [int(raw.shape[1]), int(raw.shape[0])], "clip_seed": int(seed),
        "recipe": "video.synthesize_clip: V1's recipe (M1 section 3.4), leaves over a central ellipse",
        "clip_params": clip.params, "max_seconds": max_seconds})


def overlays(result, n_frames):
    """One Overlay per source frame; a skipped frame shows the last processed frame's."""
    logged = {}
    for entry in result.log:
        if entry.get("frame") not in (None, CLIP_SCOPE):
            logged.setdefault(entry["frame"], {})[entry["node"]] = entry
    processed = [frame.index for frame in result.frames]
    shown = []
    for index in range(n_frames):
        frame = result.frames[bisect_right(processed, index) - 1]
        shown.append(overlay_text(index, frame, logged[frame.index]))
    return shown


def overlay_text(index, clip_frame, entries):
    """Source frame `index` showing processed `clip_frame`: message, top 3, own verdict and milliseconds, box."""
    shown, own = clip_frame.shown, clip_frame.verdict
    ranked, header = (shown.top3, "top 3 species") if shown.top3 else (own.top3, "top 3 species, this frame")
    top3 = tuple(f"{rank}. {name} {PERCENT.format(prob)}" for rank, (name, prob) in enumerate(ranked, 1))
    ms = sum(entries[node]["ms"] for node in PER_FRAME_NODES)
    corner = (f"frame {index} | this frame: {own.label} | {ms:.0f} ms" if index == clip_frame.index
              else f"frame {index} | skipped, showing frame {clip_frame.index} | this frame: {own.label} | {ms:.0f} ms")
    banner = (shown.message,)
    fallback = shown.genus_fallback
    if shown.label == UNKNOWN and fallback is not None:
        line = f"likeliest genus: {fallback.genus} {PERCENT.format(fallback.probability)}"
        banner += (f"{line}; {GENUS_LISTED_NOTE.format(fallback.genus)}" if fallback.who_listed else line,)
    front = entries[FRONT_END]["summary"]
    box = front.get("box")
    return Overlay(VERDICT_COLOUR[shown.label], banner, header, top3 or ("no species: no snake found",), corner,
                   None if box is None else tuple(box), front.get("box_conf"), front.get("snake_found", True))


def draw(rgb, overlay):
    """A copy of the frame with the box, the verdict banner, the top-3 panel, the corner line and the box's label."""
    image = np.array(rgb, dtype=np.uint8, copy=True)
    height, width = image.shape[:2]
    scale = max(MIN_FONT_SCALE, FONT_SCALE_AT_REFERENCE * width / REFERENCE_WIDTH)
    margin = max(EVEN, round(MARGIN_SHARE * max(height, width)))
    thickness = max(MIN_BOX_WIDTH, round(BOX_WIDTH_SHARE * max(height, width)))
    box_colour = BOX_COLOUR if overlay.snake_found else NO_SNAKE_BOX_COLOUR
    if overlay.box is not None:
        x1, y1, x2, y2 = (int(round(value)) for value in overlay.box)
        cv2.rectangle(image, (x1, y1), (x2, y2), box_colour, thickness)
    lines = _wrapped(overlay.banner, scale, width - 2 * margin)
    bottom = 2 * margin + len(lines) * _line_height(scale)
    image[:bottom] = overlay.colour
    _text(image, lines, margin, margin, scale)
    _panel(image, _wrapped([overlay.top3_header, *overlay.top3], scale, width - 2 * margin), bottom, scale, margin)
    corner_scale = scale * CORNER_SCALE
    corner = _wrapped([overlay.corner], corner_scale, width - 2 * margin)
    _panel(image, corner, height - 2 * margin - len(corner) * _line_height(corner_scale), corner_scale, margin)
    if overlay.box is not None and overlay.box_conf is not None:
        label = (f"snake {overlay.box_conf:.2f}" if overlay.snake_found
                 else f"box {overlay.box_conf:.2f}, below tau_snake")
        label_width = cv2.getTextSize(label, FONT, scale, _thickness(scale))[0][0]
        top = max(bottom, y1 - _line_height(scale) - thickness)
        _panel(image, [label], top, scale, 0, max(0, min(x2, width) - label_width), box_colour)
    return image


def _panel(image, lines, top, scale, margin, left=0, colour=TEXT_COLOUR):
    """Lines on a dark see-through panel from (left, top) down."""
    width = max(cv2.getTextSize(line, FONT, scale, _thickness(scale))[0][0] for line in lines) + 2 * margin
    bottom = min(image.shape[0], top + 2 * margin + len(lines) * _line_height(scale))
    region = image[max(0, top):bottom, max(0, left):min(image.shape[1], left + width)]
    region[:] = (region * (1 - PANEL_ALPHA) + PANEL_COLOUR * PANEL_ALPHA).astype(np.uint8)
    _text(image, lines, left + margin, top + margin, scale, colour)


def _text(image, lines, left, top, scale, colour=TEXT_COLOUR):
    descent = cv2.getTextSize(SIZING_TEXT, FONT, scale, _thickness(scale))[1]
    for row, line in enumerate(lines, 1):
        cv2.putText(image, line, (left, top + row * _line_height(scale) - descent), FONT, scale, colour,
                    _thickness(scale), cv2.LINE_AA)


def _wrapped(lines, scale, width):
    """Each line broken at spaces to fit `width` pixels."""
    wrapped = []
    for line in lines:
        current = ""
        for word in line.split(" "):
            trial = f"{current} {word}" if current else word
            if current and cv2.getTextSize(trial, FONT, scale, _thickness(scale))[0][0] > width:
                wrapped.append(current)
                current = word
            else:
                current = trial
        wrapped.append(current)
    return wrapped


def _thickness(scale):
    return max(1, round(scale * 2))


def _line_height(scale):
    (_, height), descent = cv2.getTextSize(SIZING_TEXT, FONT, scale, _thickness(scale))
    return round(height * LINE_SPACING) + descent


def write_annotated(frames, shown, fps, path, writer_factory=None):
    """Every frame with its overlay at the source frame rate, in the first of CODECS a writer opens; returns it."""
    height, width = (side - side % EVEN for side in frames[0].shape[:2])
    for codec in CODECS:
        writer = (writer_factory or cv2.VideoWriter)(str(path), cv2.VideoWriter_fourcc(*codec), fps, (width, height))
        if writer.isOpened():
            break
        writer.release()
    else:
        raise RuntimeError(f"OpenCV cannot write {path} with any of {', '.join(CODECS)}")
    for rgb, overlay in zip(frames, shown):
        writer.write(cv2.cvtColor(draw(rgb, overlay)[:height, :width], cv2.COLOR_RGB2BGR))
    writer.release()
    return codec


def insights(result, footage, run, with_text=False, codec=None):
    """What the clip showed: verdict, timing, flips, species, card, passages (text only with with_text) and speed."""
    record, clip, frames = result_record(result, with_text), result.clip, result.frames
    end = len(footage.frames) / footage.fps
    height, width = footage.frames[0].shape[:2]
    raw = video_metrics([frame.verdict.label for frame in frames], None, [frame.shown_at for frame in frames], end)
    stable_at = clip["time_to_stable"]
    by_second = {}
    for frame in frames:
        by_second.setdefault(int(frame.index // footage.fps), []).append(frame)
    timed = [entry for entry in result.log if entry.get("frame") not in (None, CLIP_SCOPE)]
    per_frame = Counter()
    for entry in timed:
        per_frame[entry["frame"]] += entry["ms"]
    return {
        "input": {**footage.source, "fps": footage.fps, "frames": len(footage.frames), "seconds": round(end, MS_DECIMALS),
                  "frame_size": [int(width), int(height)],
                  "annotated_size": [int(width - width % EVEN), int(height - height % EVEN)]},
        "model": run["model"], "decision": {**run["decision"], "smoothing": clip["method"]},
        "final": {"verdict": result.label, "message": result.verdict.message, "reason": record["reason"],
                  "reason_text": [REASON_TEXT[name] for name in result.verdict.reason if name in REASON_TEXT] or None,
                  "note": record["note"] or None, "top3": record["top3"], "genus_fallback": record["genus_fallback"],
                  "final_stable": clip["final_stable"], "majority": clip["majority"]},
        "timing": {"first_stable_verdict": next((frame.shown.label for frame in frames if frame.shown_at == stable_at),
                                                None),
                   "time_to_first_stable_s": stable_at, "first_majority_at_s": clip["first_majority_at"],
                   "stable_means": f"on screen for {STABLE_SECONDS} s"},
        "flips": {"raw": raw["flips"], "smoothed": clip["flips"], "raw_counts": clip["per_frame"],
                  "smoothed_counts": clip["shown"]},
        "species_top1": _top1_shares(frames),
        "species_top1_by_second": [{"second": second, **top[0]} for second, group in sorted(by_second.items())
                                   if (top := _top1_shares(group, 1))],
        "species_card": None if result.card is None else {**record["card"], "species": result.card.species},
        "passages": record["passages"],
        "speed": {"mean_ms": {node: round(float(np.mean([e["ms"] for e in timed if e["node"] == node])), MS_DECIMALS)
                              for node in PER_FRAME_NODES},
                  "mean_frame_ms": round(float(np.mean(list(per_frame.values()))), MS_DECIMALS),
                  "clip_ms": {e["node"]: round(e["ms"], MS_DECIMALS) for e in result.log if e.get("frame") == CLIP_SCOPE},
                  "total_ms": round(result.total_ms, MS_DECIMALS), "effective_fps": clip["effective_fps"],
                  "skipped_share": clip["skipped_share"], "processed": clip["processed"], "frames": clip["frames"],
                  "skipping": clip["skipping"]},
        "annotated": {"file": ANNOTATED_FILE, "codec": codec},
    }


def _top1_shares(frames, k=3):
    """The k species most often top 1, with their share of the processed frames."""
    counts = Counter(frame.verdict.top3[0][0] for frame in frames if frame.verdict.top3)
    return [{"species": name, "share": round(count / len(frames), 4), "frames": count}
            for name, count in counts.most_common(k)]


def run_info(run_dir, decision_file, frontend_dir, models, thresholds):
    """The model (run, encoder, front end, checkpoint hash, detector) and the decision file the demo ran with."""
    run_dir, decision_file = Path(run_dir), Path(decision_file)
    cfg = load_config(run_dir / CONFIG_FILE)
    report = json.loads(decision_file.read_text())
    detector = None
    if frontend_dir is not None:
        detector = json.loads((Path(frontend_dir) / WINNERS_FILE).read_text())[D1]["candidate"]
    return {"model": {"run_id": cfg.run_id, "encoder": cfg.encoder, "front_end": models.front_end,
                      "input_size": models.input_size, "backend": models.backend, "scorer": models.scorer,
                      "checkpoint_sha256": file_sha256(run_dir / CHECKPOINT_FILE), "detector": detector,
                      "tau_snake": models.tau_snake},
            "decision": {"file": decision_file.name, "sha256": file_sha256(decision_file),
                         "recall_bound": report.get("recall_bound", POINT),
                         "recall_bound_recorded": "recall_bound" in report,
                         "thresholds": {name: value if math.isfinite(value) else "inf"   # JSON has no infinity
                                        for name, value in asdict(thresholds).items()}}}


def insights_markdown(found):
    """A short human-readable summary of the insights."""
    final, timing, flips, speed = found["final"], found["timing"], found["flips"], found["speed"]
    source, model, decision = found["input"], found["model"], found["decision"]
    lines = [f"# SerpentGuard video demo: {model['run_id']}", "",
             f"**Final verdict: {final['message']}** ({final['verdict']}; reason {', '.join(final['reason']) or 'none'})"]
    if final["genus_fallback"]:
        fallback = final["genus_fallback"]
        lines.append(f"Likeliest genus: {fallback['genus']} {PERCENT.format(fallback['probability'])}"
                     f"{' (has WHO-listed species)' if fallback['who_listed'] else ''}")
    lines += ["", f"- First stable verdict ({timing['stable_means']}): {timing['first_stable_verdict']} at "
                  f"{timing['time_to_first_stable_s']} s; first 4-of-6 majority at {timing['first_majority_at_s']} s; "
                  f"last stable verdict {final['final_stable']}",
              f"- Flips: {flips['raw']} in the per-frame verdicts {flips['raw_counts']}, {flips['smoothed']} on the "
              f"screen {flips['smoothed_counts']} (the screen's count includes the we-don't-know shown before the "
              f"first majority)", "", "## Input", ""]
    if source["kind"] == PHOTO:
        photo = source["photo"]
        lines.append(f"- Photo `{photo['path']}` ({photo['species']}, expected {photo['expected_verdict']}); licence "
                     f"{photo['license']}, {photo['attribution']}, {photo['url']}; filmed with {source['recipe']}, "
                     f"seed {source['clip_seed']}")
    else:
        lines.append(f"- Video `{source['file']}`, source size {source['source_size']}")
    thresholds = [f"{name} {value}" if isinstance(value, str) else f"{name} {value:.4g}"
                  for name, value in decision["thresholds"].items()]
    not_recorded = " (not recorded in the file: written when the point fit was the only fit)"
    lines += [f"- {source['frames']} frames at {source['fps']:.1f} FPS ({source['seconds']} s), read at "
              f"{source['frame_size']}", "", "## Model and decision", "",
              f"- Run {model['run_id']} ({model['encoder']}), front end {model['front_end']}, input "
              f"{model['input_size']}, {model['backend']}; detector {model['detector'] or 'none'}, tau_snake "
              f"{model['tau_snake']}",
              f"- Decision file {decision['file']} (sha256 {decision['sha256'][:12]}), recall bound "
              f"{decision['recall_bound']}{'' if decision['recall_bound_recorded'] else not_recorded}, "
              f"smoothing {decision['smoothing']}",
              f"- Thresholds: {', '.join(thresholds)}", "",
              "## Species most often top 1", "", "| species | share of processed frames |", "|---|---|",
              *[f"| {entry['species']} | {entry['share']:.0%} |" for entry in found["species_top1"]], "",
              "## Species card", ""]
    card = found["species_card"]
    lines.append("None: this verdict shows no card." if card is None
                 else f"- {card['id']} ({card['species']}): {' '.join(card.get('text', card['citation']).split())}")
    lines += ["", "## Passages shown", "",
              *[f"- {p['id']} ({p['citation']}): {' '.join(p.get('text', '').split())}".rstrip(": ")
                for p in found["passages"]], "",
              "## Speed", "", "| node | mean ms per processed frame |", "|---|---|",
              *[f"| {node} | {ms:.1f} |" for node, ms in speed["mean_ms"].items()],
              f"| all four | {speed['mean_frame_ms']:.1f} |", "",
              f"Effective {speed['effective_fps']:.1f} FPS: {speed['processed']} of {speed['frames']} frames processed, "
              f"{speed['skipped_share']:.0%} skipped ({speed['skipping']}). Annotated video: "
              f"{found['annotated']['file']}, codec {found['annotated']['codec']}.", ""]
    return "\n".join(lines)


def _write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False, default=lambda value: value.item()))


def run_demo(footage, models, thresholds, lookup, out_dir, run, example=None):
    """run_pipeline on the footage; writes the annotated video, insights.json, insights.md and `example`."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    result = run_pipeline(footage.frames, models, thresholds, lookup, fps=footage.fps)
    codec = write_annotated(footage.frames, overlays(result, len(footage.frames)), footage.fps, out_dir / ANNOTATED_FILE)
    found = insights(result, footage, run, with_text=True, codec=codec)
    _write_json(out_dir / INSIGHTS_JSON, found)
    (out_dir / INSIGHTS_MD).write_text(insights_markdown(found))
    if example is not None:
        Path(example).parent.mkdir(parents=True, exist_ok=True)
        _write_json(example, insights(result, footage, run, with_text=False, codec=codec))
    return found

In [11]:
footage = (read_video(DEMO_VIDEO, DEMO_MAX_SECONDS) if DEMO_VIDEO is not None
           else photo_footage(DEMO_PHOTO, DEMO_SEED, DEMO_MAX_SECONDS))
demo = run_demo(footage, models, thresholds, lookup, DEMO_OUT,
                run_info(RUN_DIR, DECISION_FILE, FRONTEND_DIR, models, thresholds), DEMO_EXAMPLE)
final, speed = demo["final"], demo["speed"]
print(f"{final['verdict']} ({final['message']}); first stable verdict at {demo['timing']['time_to_first_stable_s']} s, "
      f"flips {demo['flips']['raw']} per frame and {demo['flips']['smoothed']} on screen")
print(f"{speed['processed']} of {speed['frames']} frames read, {speed['mean_frame_ms']:.1f} ms per processed frame; "
      f"video codec {demo['annotated']['codec']}")
print("species most often top 1:", [(entry["species"], entry["share"]) for entry in demo["species_top1"]])
print("passages shown:", [p["id"] for p in demo["passages"]])

dangerous (dangerous: WHO Category 1); first stable verdict at 0.4 s, flips 0 per frame and 1 on screen
101 of 150 frames read, 85.7 ms per processed frame; video codec avc1
species most often top 1: [('Bungarus caeruleus', 1.0)]
passages shown: ['who-110-01', 'who-119-04', 'who-118-04', 'who-119-03', 'who-120-02', 'who-120-05', 'card-029']
